<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/10_automated_causal_trace_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch transformers pandas transformer_lens accelerate

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.0 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=338e8d9c338ed932c570b3795c5488cde6acec1b44552fabf7346324f

In [2]:
import torch
import pandas as pd
import json
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer
from tqdm import tqdm

# 2. Paths (NOTE: We are now using your newly generated traced CSV)
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
TRACED_CSV_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"
OUTPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/trace_map.json"

# 3. Load Tokenizer & Model (Memory-Safe Version)
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Base HuggingFace Model to CPU (to prevent VRAM spike)...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map="cpu"  # Loads to CPU RAM first to prevent Colab crashes
)

print("Wrapping model in HookedTransformer and moving to GPU...")
model = HookedTransformer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    hf_model=hf_model,
    device="cuda",    # Safely moves the formatted model to the GPU
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False
)
model.eval()

# Delete redundant CPU model to free up System RAM
del hf_model
gc.collect()
torch.cuda.empty_cache()
print("✅ Model successfully loaded to GPU.")

Loading Tokenizer...
Loading Base HuggingFace Model to CPU (to prevent VRAM spike)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Wrapping model in HookedTransformer and moving to GPU...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Loaded pretrained model microsoft/Phi-3-mini-4k-instruct into HookedTransformer
✅ Model successfully loaded to GPU.


In [3]:
# 4. Causal Trace Engine (Updated with Dynamic Left-Padding)
def run_causal_trace(model, tokenizer, clean_prompt, corrupted_prompt, target_token_str):
    # Ensure the target token is treated as a single ID
    target_token_id = tokenizer.encode(target_token_str, add_special_tokens=False)[0]
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    # 1. Tokenize both prompts manually first
    clean_tokens = tokenizer.encode(clean_prompt, return_tensors="pt").to("cuda")
    corrupted_tokens = tokenizer.encode(corrupted_prompt, return_tensors="pt").to("cuda")

    # 2. Dynamic Left-Padding to ensure exact tensor shape match
    diff = clean_tokens.size(1) - corrupted_tokens.size(1)
    if diff > 0:
        # Clean is longer, pad corrupted
        pad_tensor = torch.full((1, diff), pad_token_id, device="cuda")
        corrupted_tokens = torch.cat([pad_tensor, corrupted_tokens], dim=1)
    elif diff < 0:
        # Corrupted is longer, pad clean
        pad_tensor = torch.full((1, -diff), pad_token_id, device="cuda")
        clean_tokens = torch.cat([pad_tensor, clean_tokens], dim=1)

    # 3. Clean run
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    clean_prob = torch.softmax(clean_logits[0, -1, :], dim=-1)[target_token_id].item()

    # 4. Corrupted run
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)
    corrupted_prob = torch.softmax(corrupted_logits[0, -1, :], dim=-1)[target_token_id].item()

    num_layers = model.cfg.n_layers
    recovery_scores = torch.zeros(num_layers, device="cuda")

    # 5. Patching function
    def patch_mlp_activation(activations, hook):
        # Because we padded, the shapes of activations and clean_cache are now guaranteed to be identical
        activations[:] = clean_cache[hook.name][:]
        return activations

    # 6. Trace loop
    for layer in range(num_layers):
        hook_point = f"blocks.{layer}.hook_mlp_out"

        with model.hooks(fwd_hooks=[(hook_point, patch_mlp_activation)]):
            patched_logits = model(corrupted_tokens)
            patched_prob = torch.softmax(patched_logits[0, -1, :], dim=-1)[target_token_id].item()

        # Recovery calculation
        recovery = (patched_prob - corrupted_prob) / (clean_prob - corrupted_prob + 1e-10)
        recovery_scores[layer] = recovery

    return recovery_scores

# 5. Mapping Execution Loop (Updated to save all 32 layer scores)
traced_df = pd.read_csv(TRACED_CSV_PATH)
trace_map = {}

print(f"\n--- Starting Automated Causal Tracing for {len(traced_df)} articles ---")

for index, row in tqdm(traced_df.iterrows(), total=len(traced_df)):
    article_id = str(row.get('id', index))

    with torch.no_grad():
        try:
            recovery_scores = run_causal_trace(
                model,
                tokenizer,
                clean_prompt=row['clean_prompt'],
                corrupted_prompt=row['corrupted_prompt'],
                target_token_str=row['target_token']
            )

            # Extract top k=2 layers dynamically for the Notebook 10b training loop
            top_k_values, top_k_indices = torch.topk(recovery_scores, k=2)
            top_layers = top_k_indices.tolist()

            # Convert the full 32-layer tensor to a standard Python list of floats
            all_scores_list = recovery_scores.tolist()

            trace_map[article_id] = {
                "text": row['text'],
                "top_layers": top_layers,
                "all_layer_scores": all_scores_list  # <-- NEW: Saves all 32 scores for thesis visualizations
            }

        except Exception as e:
            print(f"\n⚠️ Skipping Article {article_id} due to trace error: {e}")
            continue

    # Clear cache every 10 articles to prevent memory build-up
    if index % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

# 6. Save the Architectural Blueprint
with open(OUTPUT_JSON_PATH, "w") as f:
    json.dump(trace_map, f, indent=4)

print(f"\n✅ Trace mapping complete! Blueprint saved to {OUTPUT_JSON_PATH}")


--- Starting Automated Causal Tracing for 881 articles ---


100%|██████████| 881/881 [46:40<00:00,  3.18s/it]


✅ Trace mapping complete! Blueprint saved to /content/drive/MyDrive/ResearchProject/trace_map.json
